In [1]:
pip install kfp

Note: you may need to restart the kernel to use updated packages.


In [2]:
from kfp import dsl
from kfp import compiler
print("Import Dependencies")

Import Dependencies


In [3]:
pip install scikit-learn pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [30]:
from kfp import dsl
from kfp import compiler


# ============================================================
# TENSORFLOW IRIS TRAINING
# ============================================================

@dsl.component(
    base_image="tensorflow/tensorflow:2.16.1",
    packages_to_install=[
        "scikit-learn",
        "numpy"
    ]
)
def train_iris_model(
    model_output: dsl.Output[dsl.Model]
):

    import os
    import numpy as np
    import tensorflow as tf

    from sklearn.datasets import load_iris
    from sklearn.model_selection import train_test_split

    print("=" * 60)
    print("IRIS TENSORFLOW TRAINING STARTED")
    print("=" * 60)

    # --------------------------------------------------------
    # 1. TensorFlow
    # --------------------------------------------------------

    print("\n[1] TensorFlow")

    print(
        "TensorFlow version:",
        tf.__version__
    )

    # --------------------------------------------------------
    # 2. Load Iris directly
    # --------------------------------------------------------

    print("\n[2] Loading Iris dataset")

    iris = load_iris()

    X = iris.data
    y = iris.target

    print("X shape:", X.shape)
    print("Y shape:", y.shape)

    print(
        "Classes:",
        iris.target_names
    )

    # --------------------------------------------------------
    # 3. Train/Test split
    # --------------------------------------------------------

    print("\n[3] Splitting dataset")

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    print(
        "Training samples:",
        len(X_train)
    )

    print(
        "Testing samples:",
        len(X_test)
    )

    # --------------------------------------------------------
    # 4. TensorFlow Normalization
    # --------------------------------------------------------

    print("\n[4] Creating normalization layer")

    normalizer = tf.keras.layers.Normalization()

    normalizer.adapt(X_train)

    print("Normalization ready")

    # --------------------------------------------------------
    # 5. Create model
    # --------------------------------------------------------

    print("\n[5] Creating TensorFlow model")

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(4,)
        ),

        normalizer,

        tf.keras.layers.Dense(
            16,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            8,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            3,
            activation="softmax"
        )
    ])

    # --------------------------------------------------------
    # 6. Compile
    # --------------------------------------------------------

    print("\n[6] Compiling model")

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),

        loss="sparse_categorical_crossentropy",

        metrics=[
            "accuracy"
        ]
    )

    # --------------------------------------------------------
    # 7. Model summary
    # --------------------------------------------------------

    print("\n[7] Model architecture")

    model.summary()

    # --------------------------------------------------------
    # 8. Train
    # --------------------------------------------------------

    print("\n[8] Starting training")

    history = model.fit(

        X_train,

        y_train,

        validation_data=(
            X_test,
            y_test
        ),

        epochs=30,

        batch_size=16,

        verbose=1
    )

    print("\nTraining completed")

    # --------------------------------------------------------
    # 9. Evaluate
    # --------------------------------------------------------

    print("\n[9] Evaluating model")

    loss, accuracy = model.evaluate(
        X_test,
        y_test,
        verbose=0
    )

    print(
        "Test Loss:",
        loss
    )

    print(
        "Test Accuracy:",
        accuracy
    )

    # --------------------------------------------------------
    # 10. Test prediction
    # --------------------------------------------------------

    print("\n[10] Testing prediction")

    sample = np.array([
        [5.1, 3.5, 1.4, 0.2]
    ])

    prediction = model.predict(
        sample,
        verbose=0
    )

    predicted_class = int(
        np.argmax(prediction[0])
    )

    print(
        "Prediction probabilities:",
        prediction
    )

    print(
        "Predicted class:",
        predicted_class
    )

    print(
        "Predicted flower:",
        iris.target_names[predicted_class]
    )

    # --------------------------------------------------------
    # 11. Model metadata
    # --------------------------------------------------------

    print("\n[11] Setting model metadata")

    model_output.metadata["framework"] = "tensorflow"

    model_output.metadata["dataset"] = "iris"

    model_output.metadata["accuracy"] = float(
        accuracy
    )

    # --------------------------------------------------------
    # 12. Export SavedModel
    # --------------------------------------------------------

    print("\n[12] Exporting TensorFlow SavedModel")

    print(
        "Output path:",
        model_output.path
    )

    os.makedirs(
        model_output.path,
        exist_ok=True
    )

    model.export(
        model_output.path
    )

    print("\nModel exported successfully")

    # --------------------------------------------------------
    # 13. Verify
    # --------------------------------------------------------

    print("\n[13] Verifying model")

    print(
        "Model directory exists:",
        os.path.exists(
            model_output.path
        )
    )

    print(
        "Model files:"
    )

    for root, dirs, files in os.walk(
        model_output.path
    ):

        for file in files:

            print(
                os.path.join(
                    root,
                    file
                )
            )

    print("\n" + "=" * 60)
    print("IRIS TENSORFLOW TRAINING COMPLETED")
    print("=" * 60)


# ============================================================
# PIPELINE
# ============================================================

@dsl.pipeline(
    name="iris-tensorflow-simple",
    description="Simple TensorFlow Iris training pipeline"
)
def iris_tensorflow_pipeline():

    train_iris_model()


# ============================================================
# COMPILE
# ============================================================

compiler.Compiler().compile(

    pipeline_func=iris_tensorflow_pipeline,

    package_path="iris-tensorflow-simple.yaml"
)

print("\n" + "=" * 60)
print("PIPELINE YAML CREATED")
print("=" * 60)

print(
    "File: iris-tensorflow-simple.yaml"
)


PIPELINE YAML CREATED
File: iris-tensorflow-simple.yaml


In [31]:
from kfp import dsl
from kfp import compiler


# ============================================================
# COMPONENT 1
# LOAD IRIS DATA
# ============================================================

@dsl.component(
    base_image="python:3.11",
    packages_to_install=[
        "scikit-learn",
        "pandas"
    ]
)
def load_iris_data(
    dataset_output: dsl.Output[dsl.Dataset]
):

    import pandas as pd
    from sklearn.datasets import load_iris

    print("=" * 60)
    print("COMPONENT 1 — LOAD IRIS DATA")
    print("=" * 60)

    iris = load_iris()

    df = pd.DataFrame(
        iris.data,
        columns=[
            "sepal_length",
            "sepal_width",
            "petal_length",
            "petal_width"
        ]
    )

    df["target"] = iris.target

    print("\nDataset loaded")

    print("Shape:")
    print(df.shape)

    print("\nFirst rows:")
    print(df.head())

    print("\nSaving dataset...")

    df.to_csv(
        dataset_output.path,
        index=False
    )

    print(
        "Dataset saved to:",
        dataset_output.path
    )

    print("\nCOMPONENT 1 COMPLETED")


# ============================================================
# COMPONENT 2
# PREPROCESS DATA
# ============================================================

@dsl.component(
    base_image="python:3.11",
    packages_to_install=[
        "pandas",
        "scikit-learn"
    ]
)
def preprocess_data(
    input_dataset: dsl.Input[dsl.Dataset],
    train_output: dsl.Output[dsl.Dataset],
    test_output: dsl.Output[dsl.Dataset]
):

    import pandas as pd

    from sklearn.model_selection import train_test_split

    print("=" * 60)
    print("COMPONENT 2 — PREPROCESS DATA")
    print("=" * 60)

    print("\nReading dataset...")

    df = pd.read_csv(
        input_dataset.path
    )

    print(
        "Dataset shape:",
        df.shape
    )

    print("\nSplitting dataset...")

    train_df, test_df = train_test_split(
        df,
        test_size=0.20,
        random_state=42,
        stratify=df["target"]
    )

    print(
        "Train shape:",
        train_df.shape
    )

    print(
        "Test shape:",
        test_df.shape
    )

    print("\nSaving train dataset...")

    train_df.to_csv(
        train_output.path,
        index=False
    )

    print("\nSaving test dataset...")

    test_df.to_csv(
        test_output.path,
        index=False
    )

    print("\nCOMPONENT 2 COMPLETED")


# ============================================================
# COMPONENT 3
# TRAIN TENSORFLOW MODEL
# ============================================================

@dsl.component(
    base_image="tensorflow/tensorflow:2.16.1",
    packages_to_install=[
        "pandas",
        "numpy"
    ]
)
def train_tensorflow_model(
    train_dataset: dsl.Input[dsl.Dataset],
    model_output: dsl.Output[dsl.Model]
):

    import os

    import numpy as np
    import pandas as pd

    import tensorflow as tf

    print("=" * 60)
    print("COMPONENT 3 — TENSORFLOW TRAINING")
    print("=" * 60)

    print(
        "\nTensorFlow:",
        tf.__version__
    )

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    print("\nLoading training data...")

    df = pd.read_csv(
        train_dataset.path
    )

    print(
        "Training shape:",
        df.shape
    )

    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------

    feature_columns = [
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width"
    ]

    X = df[
        feature_columns
    ].values.astype(
        np.float32
    )

    y = df[
        "target"
    ].values.astype(
        np.int32
    )

    print(
        "X shape:",
        X.shape
    )

    print(
        "Y shape:",
        y.shape
    )

    # --------------------------------------------------------
    # Normalization
    # --------------------------------------------------------

    print("\nCreating normalization layer...")

    normalizer = tf.keras.layers.Normalization()

    normalizer.adapt(X)

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    print("\nCreating model...")

    model = tf.keras.Sequential([

        tf.keras.layers.Input(
            shape=(4,)
        ),

        normalizer,

        tf.keras.layers.Dense(
            16,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            8,
            activation="relu"
        ),

        tf.keras.layers.Dense(
            3,
            activation="softmax"
        )
    ])

    # --------------------------------------------------------
    # Compile
    # --------------------------------------------------------

    model.compile(

        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),

        loss="sparse_categorical_crossentropy",

        metrics=[
            "accuracy"
        ]
    )

    print("\nModel summary:")

    model.summary()

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    print("\nStarting training...")

    history = model.fit(

        X,

        y,

        epochs=30,

        batch_size=16,

        verbose=1
    )

    print("\nTraining completed")

    # --------------------------------------------------------
    # Save history as metadata
    # --------------------------------------------------------

    final_accuracy = float(
        history.history["accuracy"][-1]
    )

    print(
        "\nFinal training accuracy:",
        final_accuracy
    )

    model_output.metadata["framework"] = "tensorflow"

    model_output.metadata["dataset"] = "iris"

    model_output.metadata["accuracy"] = final_accuracy

    # --------------------------------------------------------
    # Export SavedModel
    # --------------------------------------------------------

    print("\nExporting SavedModel...")

    os.makedirs(
        model_output.path,
        exist_ok=True
    )

    model.export(
        model_output.path
    )

    print(
        "Model exported to:",
        model_output.path
    )

    print("\nCOMPONENT 3 COMPLETED")


# ============================================================
# COMPONENT 4
# EVALUATE MODEL
# ============================================================

@dsl.component(
    base_image="tensorflow/tensorflow:2.16.1",
    packages_to_install=[
        "pandas",
        "numpy"
    ]
)
def evaluate_model(
    test_dataset: dsl.Input[dsl.Dataset],
    model_input: dsl.Input[dsl.Model],
    metrics_output: dsl.Output[dsl.Metrics]
):

    import numpy as np
    import pandas as pd
    import tensorflow as tf

    print("=" * 60)
    print("COMPONENT 4 — MODEL EVALUATION")
    print("=" * 60)

    # --------------------------------------------------------
    # Load test data
    # --------------------------------------------------------

    print("\nLoading test dataset...")

    df = pd.read_csv(
        test_dataset.path
    )

    feature_columns = [
        "sepal_length",
        "sepal_width",
        "petal_length",
        "petal_width"
    ]

    X_test = df[
        feature_columns
    ].values.astype(
        np.float32
    )

    y_test = df[
        "target"
    ].values.astype(
        np.int32
    )

    print(
        "Test shape:",
        X_test.shape
    )

    # --------------------------------------------------------
    # Load model
    # --------------------------------------------------------

    print("\nLoading TensorFlow model...")

    model = tf.saved_model.load(
        model_input.path
    )

    print("Model loaded successfully")

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    print("\nRunning predictions...")

    infer = model.signatures[
        "serve"
    ]

    predictions = infer(
        tf.constant(X_test)
    )

    prediction_tensor = list(
        predictions.values()
    )[0]

    predicted_classes = tf.argmax(
        prediction_tensor,
        axis=1
    ).numpy()

    accuracy = np.mean(
        predicted_classes == y_test
    )

    print(
        "\nTest accuracy:",
        accuracy
    )

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    metrics_output.log_metric(
        "accuracy",
        float(accuracy)
    )

    print("\nMetrics recorded")

    print("\nCOMPONENT 4 COMPLETED")


# ============================================================
# COMPONENT 5
# TENSORBOARD METRICS
# ============================================================

@dsl.component(
    base_image="python:3.11",
    packages_to_install=[
        "pandas",
        "numpy"
    ]
)
def generate_metrics(
    metrics_output: dsl.Output[dsl.Metrics]
):

    import random

    print("=" * 60)
    print("COMPONENT 5 — METRICS")
    print("=" * 60)

    print("\nRecording pipeline metrics...")

    # Demo metrics.
    # Later these can come directly from training.

    accuracy = 0.95

    loss = 0.12

    metrics_output.log_metric(
        "accuracy",
        accuracy
    )

    metrics_output.log_metric(
        "loss",
        loss
    )

    print(
        "Accuracy:",
        accuracy
    )

    print(
        "Loss:",
        loss
    )

    print("\nCOMPONENT 5 COMPLETED")


# ============================================================
# COMPONENT 6
# MODEL VALIDATION
# ============================================================

@dsl.component(
    base_image="python:3.11"
)
def validate_model(
    model_input: dsl.Input[dsl.Model]
):

    import os

    print("=" * 60)
    print("COMPONENT 6 — MODEL VALIDATION")
    print("=" * 60)

    print(
        "\nModel path:",
        model_input.path
    )

    print(
        "Model exists:",
        os.path.exists(
            model_input.path
        )
    )

    print("\nModel files:")

    for root, dirs, files in os.walk(
        model_input.path
    ):

        for file in files:

            print(
                os.path.join(
                    root,
                    file
                )
            )

    saved_model = os.path.join(
        model_input.path,
        "saved_model.pb"
    )

    if not os.path.exists(
        saved_model
    ):

        raise RuntimeError(
            "saved_model.pb not found!"
        )

    print(
        "\nSavedModel validation SUCCESS"
    )

    print("\nCOMPONENT 6 COMPLETED")


# ============================================================
# COMPONENT 7
# FINAL MODEL INFORMATION
# ============================================================

@dsl.component(
    base_image="python:3.11"
)
def model_information(
    model_input: dsl.Input[dsl.Model]
):

    import os

    print("=" * 60)
    print("COMPONENT 7 — FINAL MODEL INFORMATION")
    print("=" * 60)

    print(
        "\nModel path:",
        model_input.path
    )

    print(
        "Model artifact exists:",
        os.path.exists(
            model_input.path
        )
    )

    print(
        "\nThis model is ready for:"
    )

    print(
        "1. MinIO / S3 storage"
    )

    print(
        "2. KServe deployment"
    )

    print(
        "3. REST inference"
    )

    print("\nCOMPONENT 7 COMPLETED")


# ============================================================
# PIPELINE
# ============================================================

@dsl.pipeline(
    name="iris-tensorflow-7-components",
    description="7 component TensorFlow Iris MLOps pipeline"
)
def iris_tensorflow_pipeline():

    # --------------------------------------------------------
    # Component 1
    # --------------------------------------------------------

    load_task = load_iris_data()

    # --------------------------------------------------------
    # Component 2
    # --------------------------------------------------------

    preprocess_task = preprocess_data(

        input_dataset=
            load_task.outputs[
                "dataset_output"
            ]
    )

    # --------------------------------------------------------
    # Component 3
    # --------------------------------------------------------

    train_task = train_tensorflow_model(

        train_dataset=
            preprocess_task.outputs[
                "train_output"
            ]
    )

    # --------------------------------------------------------
    # Component 4
    # --------------------------------------------------------

    evaluate_task = evaluate_model(

        test_dataset=
            preprocess_task.outputs[
                "test_output"
            ],

        model_input=
            train_task.outputs[
                "model_output"
            ]
    )

    # --------------------------------------------------------
    # Component 5
    # --------------------------------------------------------

    metrics_task = generate_metrics()

    # --------------------------------------------------------
    # Component 6
    # --------------------------------------------------------

    validate_task = validate_model(

        model_input=
            train_task.outputs[
                "model_output"
            ]
    )

    # --------------------------------------------------------
    # Component 7
    # --------------------------------------------------------

    model_information(

        model_input=
            train_task.outputs[
                "model_output"
            ]
    )


# ============================================================
# COMPILE
# ============================================================

compiler.Compiler().compile(

    pipeline_func=iris_tensorflow_pipeline,

    package_path="iris-tensorflow-7-components.yaml"
)

print("\n" + "=" * 60)
print("PIPELINE COMPILED SUCCESSFULLY")
print("=" * 60)

print(
    "YAML:",
    "iris-tensorflow-7-components.yaml"
)


PIPELINE COMPILED SUCCESSFULLY
YAML: iris-tensorflow-7-components.yaml
